In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='darkgrid')
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, MinMaxScaler
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import StratifiedKFold, train_test_split

import string
import warnings
warnings.filterwarnings('ignore')
def warn(*args, **kwargs):
    pass
warnings.warn = warn
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

from scipy.stats import skew,boxcox  # for some statistics
from scipy.special import boxcox1p
from scipy.stats import boxcox_normmax

from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from mlxtend.regressor import StackingCVRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from vecstack import stacking

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from math import sqrt
pd.set_option('display.max_columns', 100)


# 1 Data Loading

In [ ]:
raw_analysis = pd.read_csv('./analysisData.csv')
raw_scoring = pd.read_csv('./scoringData.csv')

def concat_df(train_data, test_data):
    # Returns a concatenated df of training and test set on axis 0
    return pd.concat([train_data, test_data], sort=True).reset_index(drop=True)

df_train = raw_analysis.copy()
df_test = raw_scoring.copy()
df_all = concat_df(df_train, df_test)

print('The shape of training set: {}'.format(df_train.shape))
print('The shape of test set: {}'.format(df_test.shape))
print('The shape of ALL set: {}'.format(df_all.shape))

In [ ]:
y = df_train.loc[:,['price']]

In [ ]:
y.shape

# 2 Feature Selection 

In [ ]:
columns_keep = [
                
                'accommodates','bathrooms','bedrooms','beds','bed_type',
                'cancellation_policy','cleaning_fee','extra_people','host_response_time',
                'host_has_profile_pic','guests_included',
                'review_scores_value', 'reviews_per_month', 'room_type','property_type',
                'security_deposit',
    
                'amenities', 'host_verifications','house_rules','interaction',
                'neighborhood_overview', 
                'summary','zipcode',
    
                'availability_30', 'availability_365','availability_60','availability_90',
                'calculated_host_listings_count', 'host_total_listings_count',
                'number_of_reviews','number_of_reviews_ltm'
               ]

X_all = df_all.loc[:, columns_keep]

In [ ]:
type(X_all.amenities[123])

In [ ]:

X_all.summary.isna().sum()

# 3 Data Cleaning

In [ ]:
X_all.beds = X_all.beds.fillna(X_all.beds.median())
X_all.reviews_per_month = X_all.reviews_per_month.fillna(X_all.reviews_per_month.median())

X_all['cleaning_fee_missing'] = X_all.cleaning_fee.isna()
X_all.cleaning_fee = X_all.cleaning_fee.fillna(X_all.cleaning_fee.median())

X_all.host_has_profile_pic = X_all.host_has_profile_pic.fillna(0)

X_all['security_deposit_missing']  = X_all.security_deposit.isna()
X_all.security_deposit = X_all.groupby(['beds','room_type'])['security_deposit'].apply(lambda x:x.fillna(x.median()))
X_all.security_deposit = X_all.security_deposit.fillna(X_all.security_deposit.median())

X_all['host_response_time_missing'] = X_all.host_response_time.isna()
X_all.host_response_time = X_all.host_response_time.fillna('missing')

X_all.host_total_listings_count = X_all.host_total_listings_count.fillna(X_all.host_total_listings_count.median())

X_all.house_rules = X_all.house_rules.fillna('missing')
X_all.interaction = X_all.interaction.fillna('missing')
X_all.neighborhood_overview = X_all.neighborhood_overview.fillna('missing')
X_all.number_of_reviews = X_all.number_of_reviews.fillna('missing')
X_all.summary = X_all.summary.fillna('missing')
X_all.amenities = X_all.amenities.fillna('missing')

In [ ]:
def int_transform(x):
    try: 
        if type(x)==str:
            value = int(x[0:5])
        else:
            value = int(x)
    except:
        value = 0
    return value

X_all.zipcode = X_all.zipcode.apply(int_transform)
X_all['zipcode_missing'] = (X_all.zipcode==0)

In [ ]:
X_all.isnull().sum().sum()

# 4 Feature Engineering

#### 4.1 Skewness 

In [ ]:
# #continuous_features = ['cleaning_fee','reviews_per_month','security_deposit']
# numerical_features = [
#                         'extra_people','bathrooms','guests_included',
#                         'beds','accommodates','bedrooms','cleaning_fee',
#                         'reviews_per_month',
#                         'review_scores_value',
#                         'availability_30', 'availability_365','availability_60','availability_90',
#                         'calculated_host_listings_count', 'host_total_listings_count',
#                         'number_of_reviews','number_of_reviews_ltm'
#                      ]
# numerical_features
# skewness_features = X_all.loc[:,numerical_features].skew().sort_values(ascending=False)
# skewness_features


In [ ]:
# from scipy.stats import boxcox
# for i in skewness_features.index:
#         df_tmp = X_all[i].apply(lambda x:x+0.00001)
#         try:
#             X_all[i],lmbda = boxcox(df_tmp,lmbda=None)
#         except:
#             print(i)

In [ ]:
# skewness_features = X_all.loc[:,numerical_features].skew().sort_values(ascending=False)
# skewness_features

#### 4.2 Text Processing

In [ ]:
test_features =['amenities', 'host_verifications','house_rules','interaction',
                'neighborhood_overview', 
                'summary'
               ]
for feat in test_features:
    X_all[feat] = X_all[feat].apply(lambda x: len(x))
X_all['summary'].head(10)

#### 4.3 Label Encoding

In [ ]:



cate_features = [
                 'bed_type','cancellation_policy','host_response_time',
                 'room_type', 'property_type', 'host_has_profile_pic','zipcode'
                ]
le = LabelEncoder()
for feat in cate_features:
    try:
        X_all[feat] = le.fit_transform(X_all[feat].astype(str))
    except: 
        print(feat)
    

In [ ]:
X_all.sample(3)

#### 4.4 OneHot and Not OneHot

In [ ]:
X_all1 = X_all.copy()
X_all2 = X_all.copy()

onehot_features = [
                    'bed_type', 'cancellation_policy','host_response_time',
                    'room_type', 'property_type',
                    'host_has_profile_pic','zipcode'
                  ]

for feat in onehot_features:
    dummy_col = pd.get_dummies(X_all1[feat], prefix=feat)
    X_all1 = pd.concat([X_all1, dummy_col],axis=1)

X_all1 = X_all1.drop(onehot_features, axis=1)

In [ ]:
X_all1.sample(3)

In [ ]:
# make sure features name unique
X_all2.sample(3)

#### 4.5 Max-min Scaling

In [ ]:
# scaler = MinMaxScaler(feature_range=(0,1))
# X_all1 = pd.DataFrame(data = scaler.fit_transform(X_all1),
#                       columns = X_all1.columns)
# X_all2 = pd.DataFrame(data = scaler.fit_transform(X_all2),
#                       columns = X_all2.columns)
# X_all1.sample(3)

In [ ]:
X_all2.sample(3)

In [ ]:
print(X_all2.shape)

# 5 X, X_test split
##### Now:  X_all -> X + X_test 
##### Later:......... -> (X_train+X_dev) + X_test

In [ ]:
X1, X_test1 = X_all1.iloc[0:36839], X_all1.iloc[36839:] 
X2, X_test2 = X_all2.iloc[0:36839], X_all2.iloc[36839:]
print(X1.shape)
print(X_test1.shape)
print(X2.shape)
print(X_test2.shape)

In [ ]:
X1.head()

In [ ]:
sns.distplot(y.price)

In [ ]:
filter = (X1.bathrooms < 5) & (X1.bedrooms <= 7) & (X1.amenities <= 1000) & (y.price < 400)
X1_filtered = X1[filter]
X1_filtered.shape

In [ ]:
y[filter].shape

# 6 Model Selection
##### train X_train, predict X_dev

#### 6.1 Basic GSCV setting

In [ ]:
kfolds = KFold(n_splits=5, shuffle=True, random_state=25)
param_GB = {'min_samples_leaf': [1,2,5,10,20],
            'max_depth': [3,4,5],
            'max_features': ['sqrt'],
            'learning_rate': [0.001,0.05,0.1,1],
            'criterion': ['mse'],
            'n_estimators': [100,500]
           }
GSCV = GridSearchCV(
                  GradientBoostingRegressor(),
                  param_grid = param_GB,
                  cv = kfolds,
                  scoring = 'neg_mean_squared_error',
                  n_jobs = -1
                 )


param_LGBM = {'n_estimators':[800],
              'num_leaves':[37],
              'max_depth': [6,9],
              'learning_rate': [0.05],
              'reg_alpha':[3,15,30]
             } # 60min

LGBM = GridSearchCV(
                  LGBMRegressor(objective='regression'),
                  param_grid = param_LGBM,
                  cv = kfolds,
                  scoring = 'neg_mean_squared_error',
                  n_jobs = -1
                 )

# LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
#               importance_type='split', learning_rate=0.05, max_depth=11,
#               min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
#               n_estimators=1000, n_jobs=-1, num_leaves=37,
#               objective='regression', random_state=None, reg_alpha=15,
#               reg_lambda=0.0, silent=True, subsample=1.0,
#               subsample_for_bin=200000, subsample_freq=0)
e_alphas = [1e-6,1e-5,1e-3,0.1,1,10,100] # 1:30
e_l1ratio = [0.01,0.1,0.4,0.6,0.9,0.99]
elasticnet = ElasticNetCV(
                          max_iter = 1e7, 
                          alphas = e_alphas,
                          cv=kfolds, 
                          l1_ratio=e_l1ratio
                         )

In [ ]:
kfolds

#### 6.2 Global CV for X_train X_dev split

In [ ]:
# M = []
# X_area = X1_area
X = X1[-filter]
X_test = X_test1

# LR = elasticnet.fit(X,y[filter]).best_estimator_
# sqrt(-elasticnet.best_score_)

# LR = elasticnet.fit(X,y[filter]).best_estimator_
# sqrt(-elasticnet.best_score_)


M_all = LGBM.fit(X,y[-filter]).best_estimator_
sqrt(-LGBM.best_score_)



In [ ]:
kfolds = KFold(n_splits=5, shuffle=True, random_state=25)
M = LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
                  importance_type='split', learning_rate=0.05, max_depth=11,
                  min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
                  n_estimators=1000, n_jobs=-1, num_leaves=37,
                  objective='regression', random_state=None, reg_alpha=15,
                  reg_lambda=0.0, silent=True, subsample=1.0,
                  subsample_for_bin=200000, subsample_freq=0)

cv_nmse = cross_val_score(M, X1[filter], y[filter], scoring='neg_mean_squared_error',cv=5)
print('cv score: ', sqrt(-np.mean(cv_nmse)))


M.fit(X1[filter],y[filter])
print('train score: ',sqrt(mean_squared_error(M.predict(X1[filter]), y[filter])) )
print('test score: ',sqrt(mean_squared_error(M.predict(X1[-filter]), y[-filter])) )

In [ ]:
kfolds = KFold(n_splits=30, shuffle=True, random_state=135)
cvscore = cross_val_score(LGB,X, y, scoring='neg_mean_squared_error',cv=kfolds)
print(cvscore)
print(np.mean(cvscore))

In [ ]:
GB = GSCV.best_estimator_

In [ ]:
y_final = M_all.predict(X_test)

In [ ]:
# enable = [True for i in range(60)]

In [ ]:
# y_pred_groupfit = []
# y_pred_all = []

# for row in range(0,36389):
    
#     nbg = df_train.get_value(row,'neighbourhood_cleansed')
#     p1 = M_all.predict(X.iloc[[row]])
    
#     if not (nbg in nbg_list):
#         y_pred_groupfit.append(p1)
#     else:
#         i = nbg_list.index(nbg)
#         p2 = M[i].predict(X.iloc[[row]])
#         y_pred_groupfit.append(p2)
#         p_real = y.get_value(row,'price')
#         if abs(p_real-p1) < abs(p_real-p2):
#             enable[i] = False # groupfit play worse than allfit
            
#     y_pred_all.append(p1)




In [ ]:
# y_final = []
# for row in range(0,9210):
#     nbg = df_train.get_value(row+36389,'neighbourhood_cleansed')
#     if nbg not in nbg_list:
#         y_final.append(M_all.predict(X_test.iloc[[row]]))
#     else:
#         i = nbg_list.index(nbg)
        
#         if enable[i] == True:
#             y_final.append(M[i].predict(X_test.iloc[[row]]))
#         else:
#             y_final.append(M_all.predict(X_test.iloc[[row]]))
# y_final

# 7 Final Prediction and Submit 
##### train X, predict X_test 

In [ ]:
X = X2
X_test = X_test2

M1 = LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
                   importance_type='split', learning_rate=0.01, max_depth=-1,
                   min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
                   n_estimators=10000, n_jobs=-1, num_leaves=31,
                   objective='regression', random_state=None, reg_alpha=0.0,
                   reg_lambda=0.0, silent=True, subsample=1.0,
                   subsample_for_bin=200000, subsample_freq=0)
M2 = GradientBoostingRegressor(alpha=0.9, criterion='mse', init=None,
                          learning_rate=0.1, loss='ls', max_depth=3,
                          max_features='sqrt', max_leaf_nodes=None,
                          min_impurity_decrease=0.0, min_impurity_split=None,
                          min_samples_leaf=20, min_samples_split=2,
                          min_weight_fraction_leaf=0.0, n_estimators=10000,
                          n_iter_no_change=None, presort='auto',
                          random_state=None, subsample=1.0, tol=0.0001,
                          validation_fraction=0.1, verbose=0, warm_start=False)
M3 = XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, gamma=0,
             importance_type='gain', learning_rate=0.1, max_delta_step=0,
             max_depth=3, min_child_weight=1, missing=None, n_estimators=8000,
             n_jobs=1, nthread=None, objective='reg:linear', random_state=0,
             reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
             silent=None, subsample=1, verbosity=1)
param_Stack = {'hidden_layer_sizes':[(3,),(2,),(2,2)],
               'learning_rate_init':[0.0001,0.001,0.01,0.1,0.3,0.5,0.9]
              } # 6 min
Stack = GridSearchCV(
                      MLPRegressor(),
                      param_grid = param_Stack,
                      cv = 5,
                      n_jobs = -1
                     )
Models = [M1,M2,M3]

y_Predict = []
y_test_Predict = []

for i in range(len(Models)):
        print(datetime.now())
        Models[i].fit(X, y)
        print(datetime.now(),'Predicting Models ')
        y_Predict.append(Models[i].predict(X).reshape(1,len(X))[0])
        y_test_Predict.append(Models[i].predict(X_test).reshape(1,len(X_test))[0])
        print('Model has training rmse = ',sqrt(mean_squared_error(y_Predict[i], y)))



S = np.transpose(y_Predict)
S_test = np.transpose(y_test_Predict)
Stack.fit(S, y)
print('Stack Model has training rmse = ',sqrt(mean_squared_error(Stack.predict(S), y)))
y_final = Stack.predict(S_test) #.reshape(1,len(X_test))[0])


In [ ]:
M1 = LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
                   importance_type='split', learning_rate=0.01, max_depth=-1,
                   min_child_samples=20, min_child_weight=0.001, min_split_gain=0.0,
                   n_estimators=700, n_jobs=-1, num_leaves=31,
                   objective='regression', random_state=None, reg_alpha=0.0,
                   reg_lambda=0.0, silent=True, subsample=1.0,
                   subsample_for_bin=20000, subsample_freq=0)
M3 = XGBRegressor(base_score=0.5, booster='gbtree', colsample_bylevel=1,
             colsample_bynode=1, colsample_bytree=1, gamma=0,
             importance_type='gain', learning_rate=0.1, max_delta_step=0,
             max_depth=3, min_child_weight=1, missing=None, n_estimators=2000,
             n_jobs=1, nthread=None, objective='reg:linear', random_state=0,
             reg_alpha=0, reg_lambda=1, scale_pos_weight=1, seed=None,
             silent=None, subsample=1, verbosity=1)
score = []
for i in range(1,800,50):
    M1 = LGBMRegressor(boosting_type='gbdt', class_weight=None, colsample_bytree=1.0,
                   importance_type='split', learning_rate=0.01, max_depth=-1,
                   min_child_samples=10, min_child_weight=0.001, min_split_gain=0.0,
                   n_estimators=i, n_jobs=-1, num_leaves=40,
                   objective='regression', random_state=None, reg_alpha=0.0,
                   reg_lambda=0.0, silent=True, subsample=1.0,
                   subsample_for_bin=15, subsample_freq=0)
    score.append(np.mean(cross_val_score(M1,X,y,cv=3)))
plt.plot(range(1,800,50),score, 'r*')

In [ ]:
# BESTMODEL.fit(X_BEST)
# y_final = BESTMODEL.predict(X_BEST)

result_df = pd.DataFrame(columns=['id','price'])
result_df['id'] = df_test.id
result_df['price'] = y_final
result_df = result_df.astype(int)
result_df.to_csv('submissions.csv', index=False)
result_df